# 06. Categorical Encoding and Feature Scaling

**Before this notebook:** read `notes/06_encoding_and_scaling.md`.

**Goal:** turn the text columns into numbers and put numeric columns on a comparable scale, always **fitting on the training set only**.

**Steps**
1. Load the train/test split from notebook 05
2. Encoding on a tiny example (binary, ordinal, one-hot)
3. Why scaling matters (a distance example)
4. Scaling on a tiny example
5. Apply to the real data with a `ColumnTransformer`
6. Check the result (shapes, leakage, unseen categories)
7. Exercise

## Step 1: Load the split from notebook 05

Run notebook 05 first if these files are missing.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

DATA = "../../data/processed/"
X_train = pd.read_csv(DATA + "X_train.csv")
X_test  = pd.read_csv(DATA + "X_test.csv")
y_train = pd.read_csv(DATA + "y_train.csv").squeeze()   # one-column table -> Series
y_test  = pd.read_csv(DATA + "y_test.csv").squeeze()

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (5634, 19) | X_test: (1409, 19)


**Line by line**
- We import the tools we will use: two encoders, two scalers and `ColumnTransformer`.
- `.squeeze()`: a CSV with one column loads as a one-column table. `squeeze()` turns it into a Series, which scikit-learn expects for `y`.

**Expected output:** `X_train: (5634, 19) | X_test: (1409, 19)`.

From this point the **test set is only used with `transform`, never with `fit`**.

## Step 2: Encoding on a tiny example

A small table with all three kinds of category.

In [2]:
toy = pd.DataFrame({
    "Partner":  ["Yes", "No", "No", "Yes"],                              # 2 categories
    "Contract": ["Month-to-month", "Two year", "One year", "Month-to-month"],   # has an order
    "Payment":  ["Mailed check", "Bank transfer", "Electronic check", "Mailed check"],  # no order
})
toy

,Partner,Contract,Payment
0,Yes,Month-to-month,Mailed check
1,No,Two year,Bank transfer
2,No,One year,Electronic check
3,Yes,Month-to-month,Mailed check


In [3]:
# 1) Binary: only two categories -> one 0/1 column
print("Binary (Partner):")
print((toy["Partner"] == "Yes").astype(int).tolist())

# 2) Ordinal: categories with a real order. We must give the order ourselves.
order = [["Month-to-month", "One year", "Two year"]]
ord_enc = OrdinalEncoder(categories=order)
print("\nOrdinal (Contract):")
print(ord_enc.fit_transform(toy[["Contract"]]).ravel().tolist())

# 3) One-hot: no order -> one column per category
ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(toy[["Payment"]])
print("\nOne-hot (Payment):")
print(pd.DataFrame(encoded, columns=ohe.get_feature_names_out()))

Binary (Partner):
[1, 0, 0, 1]

Ordinal (Contract):
[0.0, 2.0, 1.0, 0.0]

One-hot (Payment):
   Payment_Bank transfer  Payment_Electronic check  Payment_Mailed check
0                    0.0                       0.0                   1.0
1                    1.0                       0.0                   0.0
2                    0.0                       1.0                   0.0
3                    0.0                       0.0                   1.0


**Line by line**
- *Binary:* `== "Yes"` gives True/False, `astype(int)` makes 1/0.
- *Ordinal:* `OrdinalEncoder(categories=order)`: **we tell it the order**. Without this it would use alphabetical order, which is meaningless here. `fit_transform` learns the categories and converts them. It needs a 2-D input, so we pass `toy[["Contract"]]` with double brackets. `.ravel()` flattens the result.
- *One-hot:* `OneHotEncoder(sparse_output=False)`: returns a normal table instead of a memory-saving sparse matrix. `get_feature_names_out()` gives the new column names.

**How to read the output**
- Partner: `[1, 0, 0, 1]`.
- Contract: `[0, 2, 1, 0]`: month-to-month=0, one year=1, two year=2. The numbers carry the meaning "longer commitment".
- Payment: three columns, each row has exactly one 1.

**Our choice for the real data:** `Contract` *could* be ordinal, but its 3 groups relate to churn strongly and non-linearly (43% / 11% / 3%), so one-hot lets each group have its own effect and does not force a straight-line assumption. We **one-hot encode all multi-category columns** and use a single 0/1 column for Yes/No columns.

**What if a new category appears?**

In [4]:
new_customer = pd.DataFrame({"Payment": ["Cryptocurrency"]})    # never seen in training

strict = OneHotEncoder(sparse_output=False).fit(toy[["Payment"]])
try:
    strict.transform(new_customer)
except ValueError as e:
    print("Default encoder CRASHES:", str(e)[:80], "...")

safe = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(toy[["Payment"]])
print("With handle_unknown='ignore':", safe.transform(new_customer))

Default encoder CRASHES: Found unknown categories ['Cryptocurrency'] in column 0 during transform ...
With handle_unknown='ignore': [[0. 0. 0.]]


**How to read the output:** the default encoder raises an error on an unknown category. With `handle_unknown="ignore"` the new customer gets `[0, 0, 0]`: "none of the known categories". The model can still predict. We always set this option for real projects.

## Step 3: Why scaling matters

Two customers: A (tenure 10, charges $50) and B (tenure 60, charges $52). Which is "closer" to C (tenure 12, charges $90)?

In [5]:
A = np.array([10, 50])
B = np.array([60, 52])
C = np.array([12, 90])

print("Raw distance A-C:", round(np.linalg.norm(A - C), 1))
print("Raw distance B-C:", round(np.linalg.norm(B - C), 1))

Raw distance A-C: 40.0
Raw distance B-C: 61.2


**Line by line:** `np.linalg.norm(A - C)` is the straight-line (Euclidean) distance, the same distance KNN uses.

**How to read the output:** A is at distance about **40** from C and B at about **61**. A seems closer, **but look at why:** A and C have a nearly identical tenure (10 vs 12), but their charges differ by $40. B has a $38 charge difference but a 48-month tenure difference. Distances mix months and dollars as if they were the same unit.

Now the same calculation after standardising (using the real training data to get a realistic spread):

In [6]:
two = X_train[["tenure", "MonthlyCharges"]]
sc = StandardScaler().fit(two)

A_s, B_s, C_s = sc.transform(pd.DataFrame([A, B, C], columns=two.columns))
print("Scaled distance A-C:", round(np.linalg.norm(A_s - C_s), 2))
print("Scaled distance B-C:", round(np.linalg.norm(B_s - C_s), 2))
print("Std of tenure vs charges in training data:", round(two["tenure"].std(), 1), "vs", round(two["MonthlyCharges"].std(), 1))

Scaled distance A-C: 1.33
Scaled distance B-C: 2.33
Std of tenure vs charges in training data: 24.6 vs 30.1


**How to read the output:** the two features have similar spreads here (about 24 vs 30), so for these two columns the picture does not change very much. The real danger is `TotalCharges` (spread about 2,300), which would swamp both of them. The lesson is general: **raw distances are only meaningful if the features share a scale.** Scaling is cheap insurance for KNN, SVM and Logistic Regression.

## Step 4: Scaling on a tiny example

In [7]:
t = pd.DataFrame({"tenure": [0, 36, 72]})

print("StandardScaler:", StandardScaler().fit_transform(t).ravel().round(2).tolist())
print("MinMaxScaler:  ", MinMaxScaler().fit_transform(t).ravel().round(2).tolist())

StandardScaler: [-1.22, 0.0, 1.22]
MinMaxScaler:   [0.0, 0.5, 1.0]


**How to read the output**
- StandardScaler: `[-1.22, 0, 1.22]`: centred at 0, measured in standard deviations.
- MinMaxScaler: `[0, 0.5, 1]`: squeezed into 0-1.

We choose **StandardScaler**: it is the usual default for Logistic Regression, SVM and KNN, and it handles slightly skewed columns such as `TotalCharges` without extreme squashing.

## Step 5: Apply to the real data

We need *different* treatment for different column groups, and that is exactly what `ColumnTransformer` does.

| Group | Columns | Treatment |
|---|---|---|
| Numeric | `tenure`, `MonthlyCharges`, `TotalCharges` | `StandardScaler` |
| Already 0/1 | `SeniorCitizen` | pass through unchanged |
| Categorical | the 15 text columns | `OneHotEncoder` |

In [8]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
passthrough_cols = ["SeniorCitizen"]
categorical_cols = [c for c in X_train.columns if c not in numeric_cols + passthrough_cols]

print(len(categorical_cols), "categorical columns:", categorical_cols)

15 categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("pass", "passthrough", passthrough_cols),
    ]
)

# FIT on training data only, then transform both sets with the same fitted object.
X_train_prep = preprocessor.fit_transform(X_train)   # fit + transform
X_test_prep  = preprocessor.transform(X_test)        # transform only!

print("Before:", X_train.shape, "-> After:", X_train_prep.shape)
print("Test after:", X_test_prep.shape)

Before: (5634, 19) -> After: (5634, 40)
Test after: (1409, 40)


**Line by line**
- `categorical_cols = [...]`: everything that is not numeric or passthrough. We build this list from the data instead of typing 15 names by hand, which avoids typos.
- `ColumnTransformer(transformers=[(name, transformer, columns), ...])`: each tuple means "apply this transformer to these columns". `"passthrough"` means "leave unchanged".
- `OneHotEncoder(drop="if_binary", ...)`: Yes/No columns get **one** column. Columns with 3+ categories get one column per category. `handle_unknown="ignore"` protects against unseen categories.
- `fit_transform(X_train)`: **learns** (the means and spreads for scaling, the category lists for encoding) **and** applies, on the training data.
- `transform(X_test)`: applies what was learned. **No fitting on the test set.** This is the leakage-safe pattern from notebook 05.

**How to read the output:** 19 columns become about **40** columns. The text columns were expanded: 5 Yes/No columns stay one column each, and the 10 columns with 3-4 categories become 31 columns. Both sets have the same number of columns, which is required, because the model expects the same layout.

## Step 6: Check the result

In [10]:
feature_names = preprocessor.get_feature_names_out()
train_df = pd.DataFrame(X_train_prep, columns=feature_names)

print("First 8 column names:")
print(list(feature_names[:8]))
print()
print("Numeric columns after scaling (train):")
print(train_df.filter(like="num__").agg(["mean", "std"]).round(2))
print()
print("Any missing values after encoding?", np.isnan(X_train_prep).any())

First 8 column names:
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'cat__gender_Male', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__PhoneService_Yes', 'cat__MultipleLines_No']

Numeric columns after scaling (train):
      num__tenure  num__MonthlyCharges  num__TotalCharges
mean         -0.0                 -0.0                0.0
std           1.0                  1.0                1.0

Any missing values after encoding? False


**Line by line**
- `get_feature_names_out()`: returns the name of every output column. Prefixes show the origin: `num__` (scaled), `cat__` (encoded), `pass__` (passed through).
- `train_df.filter(like="num__")`: picks columns whose name contains `num__`.
- `.agg(["mean", "std"])`: computes both statistics for each.

**How to read the output**
- Training numeric columns: **mean 0 and standard deviation 1**, as designed.
- No missing values (`False`), so the matrix is ready for a model.
- One-hot column names such as `cat__Contract_Month-to-month` show exactly which category each column means.

Now the same check on the **test** set: it should be *close to*, but not exactly, 0 and 1.

In [11]:
test_df = pd.DataFrame(X_test_prep, columns=feature_names)
print(test_df.filter(like="num__").agg(["mean", "std"]).round(2))

      num__tenure  num__MonthlyCharges  num__TotalCharges
mean        -0.02                -0.03              -0.04
std          1.00                 0.99               0.97


**How to read the output:** the test means are near 0 and the standard deviations near 1, but not exact. **That is correct.** The test data was transformed using the *training* numbers. Exactly 0 and 1 would be a sign of a leak.

Finally, a sanity check on how the Yes/No encoding came out:

In [12]:
print(X_train["Partner"].head(3).tolist())
print(train_df["cat__Partner_Yes"].head(3).tolist())

['No', 'Yes', 'Yes']
[0.0, 1.0, 1.0]


**How to read the output:** `Partner` = Yes became 1 and No became 0 in the single column `cat__Partner_Yes`. The column name tells you which category is "1".

**What we do with this:** we do **not** save these matrices. In notebook 08 the `ColumnTransformer` goes inside a **Pipeline** together with the model, so that encoding and scaling are re-fitted automatically on each training fold during cross-validation, and the final saved model contains its own preprocessing.

## Summary

| Group | Columns | Treatment | Fitted on |
|---|---|---|---|
| Numeric (3) | tenure, MonthlyCharges, TotalCharges | StandardScaler | Train only |
| Yes/No (5) | gender, Partner, Dependents, PhoneService, PaperlessBilling | One-hot, `drop="if_binary"` | Train only |
| Multi-category (10) | InternetService, Contract, PaymentMethod, ... | One-hot, `handle_unknown="ignore"` | Train only |
| Already 0/1 | SeniorCitizen | Passthrough | None |

**Next:** `07_feature_engineering_and_selection`: create new useful features and decide which weak ones to remove.

---
## Exercise

1. Replace `StandardScaler` with `MinMaxScaler` in the preprocessor. What are the min and max of the *scaled test* columns? Are they exactly 0 and 1?
2. Change `drop="if_binary"` to `drop="first"`. How many columns do you get now? Why is it fewer?
3. Add an extra row to `X_test` (copy a row and change its `PaymentMethod` to `"Cryptocurrency"`). Transform it. Does it crash? What does `handle_unknown="ignore"` do to that row's `PaymentMethod_...` columns?
4. **Think:** you are going to train a Random Forest only. Is scaling necessary? Would it hurt?

In [13]:
# Exercise 1

In [14]:
# Exercise 2